# The Training Loop, Line by Line

LLM Training Lab, module 5. FROM SCRATCH on char-level Tiny Shakespeare. Run the cells in order. Steps 1 to 8 and 10 run on a laptop CPU. Step 9 needs a GPU.

<!-- llt-run-help -->
## How to run this notebook

- **Check the lab card** on the module page first. It says whether this lab needs a GPU and how long it took us.
- **Kaggle** (where we recorded most runs): use the lab page's *Open on Kaggle* button. If the lab needs a GPU, open the notebook settings and pick *Accelerator: GPU T4 x2* (not P100). Switch *Internet* on so the install cell and the downloads work. Kaggle asks you to verify a phone number once before it allows a GPU or internet.
- **Colab**: use the lab page's *Open in Colab* button. If the lab needs a GPU, pick *Runtime > Change runtime type > T4 GPU*.
- **Hugging Face**: no account or token is needed. Our checkpoints and data files download from the public repo [aimlcompanion/llm-training-lab](https://huggingface.co/aimlcompanion/llm-training-lab).
- **Run the cells in order.** Code cell numbers match the lab page. Your numbers should land close to ours, not match exactly: GPU runs are not bit-for-bit repeatable.

In [ ]:
# Runtime check. Run this first. It prints the fix for each problem it finds.
import platform, shutil, sys
print("Python", platform.python_version())
if sys.version_info < (3, 11):
    print("FIX: use Python 3.11 or later, 3.12 recommended. On Colab, keep the default runtime.")
try:
    import numpy, torch
    print("torch", torch.__version__, "| numpy", numpy.__version__)
    print("Our CPU figures came from torch 2.15.0 and numpy 2.5.4. Other versions can change the last digits of the losses.")
    if torch.cuda.is_available():
        cap = torch.cuda.get_device_capability()
        print("GPU", torch.cuda.get_device_name(), "capability", cap)
        if cap < (7, 0):
            print("FIX: this GPU is too old for current PyTorch wheels. On Kaggle choose GPU T4 x2, not P100.")
    else:
        print("No GPU. Steps 1 to 8 and 10 run on this CPU. Step 9 needs a GPU: Colab > Runtime > Change runtime type > T4 GPU.")
except ImportError:
    print("FIX: install PyTorch, pip install torch")
try:
    import psutil
    print(f"RAM {psutil.virtual_memory().total / 1e9:.1f} GB")
except ImportError:
    pass
print(f"Free disk {shutil.disk_usage('.').free / 1e9:.1f} GB (the data and checkpoints need under 0.1 GB)")

## Step 1. Build a memmapped token pipeline with a resumable sampler

In [ ]:
import os
import urllib.request
import numpy as np
import torch

URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"


def prepare(data_dir="data"):
    # Download once, map each character to an id, write uint8 token files.
    os.makedirs(data_dir, exist_ok=True)
    path = os.path.join(data_dir, "input.txt")
    if not os.path.exists(path):
        urllib.request.urlretrieve(URL, path)
    text = open(path, encoding="utf-8").read()
    chars = sorted(set(text))
    stoi = {c: i for i, c in enumerate(chars)}
    if not os.path.exists(os.path.join(data_dir, "val.bin")):
        ids = np.array([stoi[c] for c in text], dtype=np.uint8)
        n = int(0.9 * len(ids))
        ids[:n].tofile(os.path.join(data_dir, "train.bin"))
        ids[n:].tofile(os.path.join(data_dir, "val.bin"))
    return chars


class Sampler:
    # Seeded batch order over a memmapped token file. Its position is two integers.
    def __init__(self, path, block, batch, seed):
        self.data = np.memmap(path, dtype=np.uint8, mode="r")
        self.block, self.batch, self.seed = block, batch, seed
        self.n = (len(self.data) - 1 - block) // block
        self.epoch, self.pos = 0, 0
        self._shuffle()

    def _shuffle(self):
        # The order depends only on (seed, epoch), so it can be rebuilt after a restart.
        rng = np.random.default_rng([self.seed, self.epoch])
        offset = int(rng.integers(self.block))
        self.starts = offset + self.block * rng.permutation(self.n)

    def next_batch(self):
        if self.pos + self.batch > self.n:
            self.epoch, self.pos = self.epoch + 1, 0
            self._shuffle()
        s = self.starts[self.pos:self.pos + self.batch]
        self.pos += self.batch
        x = np.stack([self.data[i:i + self.block] for i in s]).astype(np.int64)
        y = np.stack([self.data[i + 1:i + 1 + self.block] for i in s]).astype(np.int64)
        return torch.from_numpy(x), torch.from_numpy(y)

    def state_dict(self):
        return {"seed": self.seed, "epoch": self.epoch, "pos": self.pos}

    def load_state_dict(self, state):
        assert state["seed"] == self.seed, "sampler seed differs from the checkpoint"
        self.epoch, self.pos = state["epoch"], state["pos"]
        self._shuffle()

In [ ]:
chars = prepare()
train_ids = np.memmap("data/train.bin", dtype=np.uint8, mode="r")
val_ids = np.memmap("data/val.bin", dtype=np.uint8, mode="r")
print("vocab", len(chars), "| train tokens", len(train_ids), "| val tokens", len(val_ids))

a = Sampler("data/train.bin", block=64, batch=12, seed=0)
print("windows per epoch", a.n, "| batches per epoch", a.n // a.batch)
for _ in range(5):
    a.next_batch()
saved = a.state_dict()
x_a, y_a = a.next_batch()

b = Sampler("data/train.bin", block=64, batch=12, seed=0)
b.load_state_dict(saved)
x_b, y_b = b.next_batch()
print("saved position", saved, "| same 6th batch after restore", torch.equal(x_a, x_b) and torch.equal(y_a, y_b))
print("targets are inputs shifted by one", torch.equal(x_a[:, 1:], y_a[:, :-1]))

## Step 2. Write the model and the AdamW decay groups

In [ ]:
import math
import torch.nn as nn
import torch.nn.functional as F


class Block(nn.Module):
    def __init__(self, d, n_head, dropout):
        super().__init__()
        self.n_head, self.dropout = n_head, dropout
        self.ln1, self.ln2 = nn.LayerNorm(d, bias=False), nn.LayerNorm(d, bias=False)
        self.qkv, self.proj = nn.Linear(d, 3 * d, bias=False), nn.Linear(d, d, bias=False)
        self.fc, self.out = nn.Linear(d, 4 * d, bias=False), nn.Linear(4 * d, d, bias=False)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(self.ln1(x)).split(C, dim=2)
        q, k, v = (t.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) for t in (q, k, v))
        p = self.dropout if self.training else 0.0
        a = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=p)
        x = x + self.drop(self.proj(a.transpose(1, 2).reshape(B, T, C)))
        return x + self.drop(self.out(F.gelu(self.fc(self.ln2(x)))))


class GPT(nn.Module):
    def __init__(self, vocab, block, n_layer, n_head, d, dropout=0.0):
        super().__init__()
        self.tok, self.pos = nn.Embedding(vocab, d), nn.Embedding(block, d)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(d, n_head, dropout) for _ in range(n_layer)])
        self.ln_f, self.head = nn.LayerNorm(d, bias=False), nn.Linear(d, vocab, bias=False)
        for name, p in self.named_parameters():
            if p.dim() == 2:
                # GPT-2 init: std 0.02, and the residual output layers scaled by 1/sqrt(2L)
                std = 0.02 / math.sqrt(2 * n_layer) if name.endswith(("proj.weight", "out.weight")) else 0.02
                nn.init.normal_(p, 0.0, std)

    def forward(self, idx):
        x = self.drop(self.tok(idx) + self.pos(torch.arange(idx.shape[1], device=idx.device)))
        for blk in self.blocks:
            x = blk(x)
        return self.head(self.ln_f(x))


def param_groups(model, weight_decay):
    # Decay the weight matrices of linear layers only. Norm gains and embeddings keep their size.
    decay, no_decay = [], []
    for module in model.modules():
        for name, p in module.named_parameters(recurse=False):
            if isinstance(module, nn.Linear) and name == "weight":
                decay.append(p)
            else:
                no_decay.append(p)
    return [{"params": decay, "weight_decay": weight_decay}, {"params": no_decay, "weight_decay": 0.0}]

In [ ]:
SMALL = dict(n_layer=4, n_head=4, d=128, block=64)
T2 = dict(n_layer=6, n_head=6, d=384, block=256)

for name, size in [("small (CPU)", SMALL), ("10M (T4)", T2)]:
    model = GPT(len(chars), **size)
    groups = param_groups(model, weight_decay=0.1)
    count = lambda g: sum(p.numel() for p in g["params"])
    print(name, "| parameters", sum(p.numel() for p in model.parameters()),
          "| decay", len(groups[0]["params"]), "tensors", count(groups[0]),
          "| no decay", len(groups[1]["params"]), "tensors", count(groups[1]))

## Step 3. Write warmup plus cosine and WSD schedules

In [ ]:
from torch.optim.lr_scheduler import LambdaLR


def lr_cosine(step, peak, warmup, total, min_ratio=0.1):
    if step < warmup:
        return peak * (step + 1) / warmup
    progress = min(1.0, (step - warmup) / max(1, total - warmup))
    return peak * (min_ratio + (1 - min_ratio) * 0.5 * (1 + math.cos(math.pi * progress)))


def lr_wsd(step, peak, warmup, total, min_ratio=0.1, decay_frac=0.2):
    # Warmup, then a flat stable phase, then a linear decay over the last decay_frac of steps.
    decay_start = int(total * (1 - decay_frac))
    if step < warmup:
        return peak * (step + 1) / warmup
    if step < decay_start:
        return peak
    progress = min(1.0, (step - decay_start) / max(1, total - decay_start))
    return peak * (1 - (1 - min_ratio) * progress)


def make_scheduler(opt, cfg):
    fn = lr_wsd if cfg["schedule"] == "wsd" else lr_cosine
    # LambdaLR multiplies the optimizer's base lr (cfg["lr"]) by this factor at every step.
    return LambdaLR(opt, lambda s: fn(s, cfg["lr"], cfg["warmup"], cfg["steps"]) / cfg["lr"])

In [ ]:
peak, warmup, total = 1e-3, 100, 2000
print("step   cosine     wsd")
for s in [0, 50, 99, 100, 1000, 1599, 1600, 1800, 1999]:
    print(f"{s:>4}  {lr_cosine(s, peak, warmup, total):.2e}  {lr_wsd(s, peak, warmup, total):.2e}")

# The scheduler object holds its position, so it goes into the checkpoint.
opt = torch.optim.AdamW(GPT(len(chars), **SMALL).parameters(), lr=peak)
sched = make_scheduler(opt, dict(schedule="cosine", lr=peak, warmup=warmup, steps=total))
for _ in range(150):
    opt.step()
    sched.step()
print("after 150 steps", sched.state_dict()["last_epoch"], "| lr", f"{opt.param_groups[0]['lr']:.3e}")

## Step 4. Accumulate gradients and clip the global norm

In [ ]:
def grads_of(model):
    return torch.cat([p.grad.flatten() for p in model.parameters()])


def global_norm(model):
    # One norm over every gradient in the model, as if they were one long vector
    return math.sqrt(sum(float(p.grad.pow(2).sum()) for p in model.parameters()))


torch.manual_seed(0)
model = GPT(len(chars), **SMALL)
x, y = Sampler("data/train.bin", block=64, batch=48, seed=0).next_batch()
loss_fn = lambda xb, yb: F.cross_entropy(model(xb).flatten(0, 1), yb.flatten())

model.zero_grad()
loss_fn(x, y).backward()                      # one batch of 48
full = grads_of(model)

N = 4                                          # 4 micro-batches of 12
for divide in [True, False]:
    model.zero_grad()
    for xb, yb in zip(x.chunk(N), y.chunk(N)):
        loss = loss_fn(xb, yb)
        (loss / N if divide else loss).backward()
    acc = grads_of(model)
    print("divide by N" if divide else "no divide  ", "| max diff from full batch", f"{(acc - full).abs().max().item():.1e}",
          "| norm ratio", round(acc.norm().item() / full.norm().item(), 4))

model.zero_grad()
loss_fn(x, y).backward()
norm = global_norm(model)
returned = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
print("global norm", round(norm, 4), "| clip_grad_norm_ returned", round(returned.item(), 4),
      "| norm after clipping to 1.0", round(global_norm(model), 4))

## Step 5. Save everything a resume needs

In [ ]:
import random


def save_checkpoint(path, model, opt, sched, scaler, sampler, step, cfg):
    state = {
        "model": model.state_dict(), "optimizer": opt.state_dict(), "scheduler": sched.state_dict(),
        "scaler": scaler.state_dict(), "sampler": sampler.state_dict(), "step": step, "cfg": cfg,
        "chars": chars,
        "rng": {"python": random.getstate(), "numpy": np.random.get_state(), "torch": torch.get_rng_state(),
                "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None},
    }
    torch.save(state, path + ".tmp")
    os.replace(path + ".tmp", path)  # a session that dies mid-save leaves the old file intact


def load_checkpoint(path, model, opt, sched, scaler, sampler, skip=()):
    # weights_only=False because the RNG states are Python and NumPy objects. Load only your own files.
    state = torch.load(path, map_location="cpu", weights_only=False)
    model.load_state_dict(state["model"])
    if "optimizer" not in skip:
        opt.load_state_dict(state["optimizer"])
    if "scheduler" not in skip:
        sched.load_state_dict(state["scheduler"])
    scaler.load_state_dict(state["scaler"])
    if "sampler" not in skip:
        sampler.load_state_dict(state["sampler"])
    if "rng" not in skip:
        random.setstate(state["rng"]["python"])
        np.random.set_state(state["rng"]["numpy"])
        torch.set_rng_state(state["rng"]["torch"])
        if state["rng"]["cuda"] is not None and torch.cuda.is_available():
            torch.cuda.set_rng_state_all(state["rng"]["cuda"])
    return state["step"]

In [ ]:
cfg = dict(SMALL, lr=1e-3, warmup=100, steps=2000, schedule="cosine")
torch.manual_seed(0)
model = GPT(len(chars), **SMALL)
opt = torch.optim.AdamW(param_groups(model, 0.1), lr=cfg["lr"])
sched = make_scheduler(opt, cfg)
scaler = torch.amp.GradScaler("cpu", enabled=False)
sampler = Sampler("data/train.bin", block=64, batch=12, seed=0)
for _ in range(3):  # three updates, so the optimizer has state to save
    x, y = sampler.next_batch()
    F.cross_entropy(model(x).flatten(0, 1), y.flatten()).backward()
    opt.step()
    opt.zero_grad()
    sched.step()

save_checkpoint("demo.pt", model, opt, sched, scaler, sampler, 3, cfg)
state = torch.load("demo.pt", weights_only=False)
print("keys", sorted(state), "| file", round(os.path.getsize("demo.pt") / 1e6, 2), "MB")
print("AdamW keeps per tensor", sorted(state["optimizer"]["state"][0]))
print("sampler", state["sampler"], "| scheduler position", state["scheduler"]["last_epoch"])

## Step 6. Write the training loop and train the small model

In [ ]:
import json
import time
from contextlib import nullcontext

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def pick_dtype():
    # T4 is compute capability 7.5: fp16 tensor cores, no bf16 ones, although
    # torch.cuda.is_bf16_supported() returns True there. Decide by capability.
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.get_device_capability() >= (8, 0) else torch.float16


@torch.no_grad()
def evaluate(model, cfg, dtype):
    # Mean loss over every window of the validation file, in a fixed order.
    model.eval()
    val = np.memmap("data/val.bin", dtype=np.uint8, mode="r")
    T = cfg["block"]
    starts = np.arange(0, len(val) - T - 1, T)
    total, count = 0.0, 0
    for i in range(0, len(starts), 64):
        s = starts[i:i + 64]
        x = torch.from_numpy(np.stack([val[j:j + T] for j in s]).astype(np.int64)).to(device)
        y = torch.from_numpy(np.stack([val[j + 1:j + 1 + T] for j in s]).astype(np.int64)).to(device)
        with autocast(dtype):
            logits = model(x)
        total += F.cross_entropy(logits.float().flatten(0, 1), y.flatten(), reduction="sum").item()
        count += y.numel()
    model.train()
    return total / count


def autocast(dtype):
    return torch.autocast(device.type, dtype=dtype) if dtype != torch.float32 else nullcontext()


def train(cfg, log_path=None):
    torch.manual_seed(cfg["seed"])
    dtype = pick_dtype()
    model = GPT(len(chars), cfg["block"], cfg["n_layer"], cfg["n_head"], cfg["d"], cfg["dropout"]).to(device)
    opt = torch.optim.AdamW(param_groups(model, cfg["wd"]), lr=cfg["lr"], betas=(0.9, 0.99))
    sched = make_scheduler(opt, cfg)
    scaler = torch.amp.GradScaler(device.type, enabled=(dtype == torch.float16))
    sampler = Sampler("data/train.bin", cfg["block"], cfg["micro_batch"], cfg["seed"])
    step = 0
    if cfg.get("resume"):
        step = load_checkpoint(cfg["ckpt"], model, opt, sched, scaler, sampler, skip=cfg.get("skip", ()))
    elif log_path:
        open(log_path, "w").close()
    N, history, t0 = cfg["accum"], [], time.time()
    while step < cfg.get("stop_at", cfg["steps"]):
        opt.zero_grad(set_to_none=True)
        micro_losses = []
        for _ in range(N):
            x, y = (t.to(device) for t in sampler.next_batch())
            with autocast(dtype):
                loss = F.cross_entropy(model(x).float().flatten(0, 1), y.flatten())
            scaler.scale(loss / N if cfg.get("divide_loss", True) else loss).backward()
            micro_losses.append(loss.item())
            if cfg.get("sched_per_micro"):
                sched.step()                     # broken on purpose in step 10
        scaler.unscale_(opt)                     # clip the true gradients, not the scaled ones
        norm = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["clip"]).item()
        lr = opt.param_groups[0]["lr"]
        scaler.step(opt)                         # skips the update if fp16 gradients overflowed
        scaler.update()
        if not cfg.get("sched_per_micro"):
            sched.step()
        step += 1
        row = {"step": step, "trainLoss": sum(micro_losses) / N, "gradNorm": norm, "lr": lr}
        if step % cfg["eval_every"] == 0 or step == cfg["steps"]:
            row["valLoss"] = evaluate(model, cfg, dtype)
            row["elapsedSec"] = round(time.time() - t0, 1)
        history.append(row)
        if log_path:
            with open(log_path, "a") as f:
                print(json.dumps(row), file=f)
        if cfg.get("ckpt") and step % cfg.get("ckpt_every", 10**9) == 0:
            save_checkpoint(cfg["ckpt"], model, opt, sched, scaler, sampler, step, cfg)
    return model, history

In [ ]:
torch.set_num_threads(4)
if device.type == "cpu":
    torch.use_deterministic_algorithms(True)

CPU_CFG = dict(SMALL, dropout=0.0, micro_batch=12, accum=1, steps=2000, lr=1e-3, warmup=100,
               schedule="cosine", wd=0.1, clip=1.0, seed=0, eval_every=250)
print("device", device, "| dtype", pick_dtype())
model, history = train(dict(CPU_CFG, ckpt="small-s0.pt", ckpt_every=2000), log_path="small-s0.jsonl")
for row in history:
    if "valLoss" in row:
        print("step", row["step"], "| train", round(row["trainLoss"], 4), "| val", round(row["valLoss"], 4),
              "| grad norm", round(row["gradNorm"], 3), "| lr", f"{row['lr']:.2e}")

In [ ]:
@torch.no_grad()
def generate(path, prompt, n=300, temperature=0.8, seed=0):
    # Rebuild the model from the checkpoint alone: its config, its vocabulary, its weights.
    state = torch.load(path, map_location="cpu", weights_only=False)
    c, vocab = state["cfg"], state["chars"]
    m = GPT(len(vocab), c["block"], c["n_layer"], c["n_head"], c["d"]).to(device).eval()
    m.load_state_dict(state["model"])
    idx = torch.tensor([[vocab.index(ch) for ch in prompt]], device=device)
    g = torch.Generator(device=device).manual_seed(seed)
    for _ in range(n):
        logits = m(idx[:, -c["block"]:])[:, -1].float() / temperature
        idx = torch.cat([idx, torch.multinomial(F.softmax(logits, -1), 1, generator=g)], 1)
    return "".join(vocab[i] for i in idx[0].tolist())


print(generate("small-s0.pt", "ROMEO:"))

## Step 7. Prove that a resumed run matches a straight run

In [ ]:
R = dict(CPU_CFG, dropout=0.1, steps=200, eval_every=100, ckpt="resume.pt", ckpt_every=100)

_, straight = train(R)                                   # 200 steps in one go
_, first = train(dict(R, stop_at=100))                   # 100 steps, checkpoint at step 100
_, second = train(dict(R, resume=True, ckpt_every=10**9))  # new objects, load, run steps 101 to 200


def compare(a, b):
    diff = max(abs(x["trainLoss"] - y["trainLoss"]) for x, y in zip(a, b))
    return diff, b[-1]["valLoss"] - a[-1]["valLoss"]


diff, dval = compare(straight, first + second)
print("full checkpoint | max train-loss diff", diff, "| val diff at step 200", dval)

for missing in ["rng", "sampler", "scheduler", "optimizer"]:
    _, part = train(dict(R, resume=True, ckpt_every=10**9, skip=(missing,)))
    diff, dval = compare(straight, first + part)
    print(f"without {missing:<9} | max train-loss diff {diff:.4f} | val diff at step 200 {dval:+.4f}",
          f"| loss at step 101 {part[0]['trainLoss']:.4f} vs {straight[100]['trainLoss']:.4f}")

## Step 8. Measure seed spread and test whether a difference is real

In [ ]:
final = {"cosine": [], "wsd": []}
for schedule in final:
    for seed in range(3):
        _, hist = train(dict(CPU_CFG, schedule=schedule, seed=seed))
        final[schedule].append(hist[-1]["valLoss"])
        print(schedule, "seed", seed, "| final val", round(hist[-1]["valLoss"], 4))

In [ ]:
def mean_se(xs):
    xs = np.asarray(xs, dtype=float)
    return xs.mean(), xs.std(ddof=1), xs.std(ddof=1) / np.sqrt(len(xs))


def compare(a, b, name_a, name_b):
    ma, sa, ea = mean_se(a)
    mb, sb, eb = mean_se(b)
    diff, se_diff = mb - ma, np.sqrt(ea ** 2 + eb ** 2)
    print(f"{name_a}: mean {ma:.4f}, std {sa:.4f}, SE {ea:.4f}")
    print(f"{name_b}: mean {mb:.4f}, std {sb:.4f}, SE {eb:.4f}")
    verdict = "a real difference" if abs(diff) > 2 * se_diff else "not a claim, within 2 SE"
    print(f"difference {diff:+.4f} | SE of difference {se_diff:.4f} | {abs(diff) / se_diff:.1f} SE | {verdict}")


compare(final["cosine"], final["wsd"], "cosine", "wsd")

## Step 9. Sweep the learning rate on the 10M model (needs a GPU)

In [ ]:
T2_CFG = dict(T2, dropout=0.2, micro_batch=64, accum=1, steps=2000, warmup=100, schedule="cosine",
              wd=0.1, clip=1.0, eval_every=250)
LRS = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2]
SEEDS = [0]              # one seed per rate on a free T4; [0, 1, 2] for the full sweep
print("device", device, "| dtype", pick_dtype())

results = []
if device.type != "cuda":
    print("This step needs a GPU. Use the table of our runs below it.")
else:
    for lr in LRS:
        for seed in SEEDS:
            _, hist = train(dict(T2_CFG, lr=lr, seed=seed), log_path=f"t2-lr{lr:.0e}-s{seed}.jsonl")
            results.append({"lr": lr, "seed": seed, "final": hist[-1]["valLoss"]})
            print(f"lr {lr:.0e} seed {seed} | final val {hist[-1]['valLoss']:.4f}")

In [ ]:
print("lr       runs   mean     std")
for lr in LRS:
    runs = [r["final"] for r in results if r["lr"] == lr]
    xs = [x for x in runs if x == x]          # drop NaN, a run that diverged
    if not runs:
        print(f"{lr:.0e}   0")
    elif not xs:
        print(f"{lr:.0e}   {len(runs)}      every run diverged")
    else:
        std = np.std(xs, ddof=1) if len(xs) > 1 else 0.0
        print(f"{lr:.0e}   {len(xs)}      {np.mean(xs):.4f}   {std:.4f}")

## Step 10. Break the loop on purpose

In [ ]:
ACC = dict(CPU_CFG, micro_batch=3, accum=4)    # 4 micro-batches of 3, the same 12 sequences per update
RUNS = {
    "control, accumulate 4 x 3": ACC,
    "loss not divided by 4": dict(ACC, divide_loss=False),
    "scheduler stepped per micro-batch": dict(ACC, sched_per_micro=True),
    "learning rate 1e-1": dict(CPU_CFG, lr=1e-1),
}
for name, cfg in RUNS.items():
    _, hist = train(cfg)
    vals = {r["step"]: round(r["valLoss"], 4) for r in hist if "valLoss" in r}
    clipped = sum(r["gradNorm"] > cfg["clip"] for r in hist)
    print(f"{name:<34} val {vals} | clipped steps {clipped} | lr at step 600 {hist[599]['lr']:.2e}")